# `ar-db-handler` playground

A guided tour through every public helper in `ar_db_handler`.

> **This notebook never touches a real database.** Every cell writes to
> a fresh `tempfile.TemporaryDirectory()` that lives only for the
> lifetime of this kernel. When the kernel shuts down, the temp dir is
> deleted. Your `filings.db` and `metrics.db` on disk are unaffected.

We'll walk through:

1. Initialising both databases
2. Recording a scraper run + its workers
3. Writing companies, filings, and filing files
4. The three statuses (`PENDING`, `SCRAPED`, `FAILED`) and how upsert
   semantics differ between them
5. `AlreadyScrapedError` and `force=True`
6. Reading data back: `get_filing`, `get_filing_file`, `list_companies`
7. `get_scraped_pairs()` — the evaluator's discovery query
8. Writing evaluation metrics into `metrics.db`

## 0. Setup — a throwaway working directory

Everything below this point uses `WORKDIR`, a temporary path. The
`TemporaryDirectory` object is held in a module-level variable so it
isn't garbage-collected (and thus deleted) before we're done with it.

In [ ]:
import json
import sqlite3
import tempfile
import uuid
from datetime import datetime, timezone
from pathlib import Path

import ar_db_handler as ardb

# A throwaway directory. Held in a name so it isn't GC'd mid-notebook.
_TMP = tempfile.TemporaryDirectory(prefix="ar_db_playground_")
WORKDIR = Path(_TMP.name)

FILINGS_PATH = WORKDIR / "filings.db"
METRICS_PATH = WORKDIR / "metrics.db"

print(f"ar_db_handler v{ardb.__version__}")
print(f"Working dir : {WORKDIR}")
print(f"filings.db  : {FILINGS_PATH}")
print(f"metrics.db  : {METRICS_PATH}")

A small helper to ISO-format timestamps the way SQLite expects:

In [ ]:
def now_iso() -> str:
    """UTC timestamp, ISO 8601, second precision."""
    return datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")


now_iso()

## 1. Initialise both databases

`init_filings_db(path)` and `init_metrics_db(path)` each return an open
`sqlite3.Connection`. They:

- create the file if it doesn't exist,
- apply the schema (`CREATE TABLE IF NOT EXISTS` everywhere — so calling
  on an existing DB is a safe no-op),
- enable WAL journal mode,
- turn on foreign-key enforcement.

The caller owns the connection and is responsible for closing it. The
notebook keeps both open until the end.

In [ ]:
filings = ardb.init_filings_db(FILINGS_PATH)
metrics = ardb.init_metrics_db(METRICS_PATH)

# Sanity check: the standard pragmas are on.
for name, conn in [("filings", filings), ("metrics", metrics)]:
    jm = conn.execute("PRAGMA journal_mode").fetchone()[0]
    fk = conn.execute("PRAGMA foreign_keys").fetchone()[0]
    print(f"{name:8s}  journal_mode={jm!r}  foreign_keys={fk}")

In [ ]:
# Show the tables that were created.
def tables(conn):
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    return [r["name"] for r in rows]


print("filings.db tables:", tables(filings))
print("metrics.db tables:", tables(metrics))

## 2. Record a scraper run + its workers

`scraper_runs` is a slightly unusual table: it holds **two kinds of
rows**.

- A *parent* row, with `worker_id IS NULL`, carries the run-level
  metadata (`country`, `config`, `worker_count`).
- One row per worker, with `worker_id` set and `country`/`config` NULL.
  Worker rows reference the parent via `parent_run_id`.

`upsert_run` writes the parent. `upsert_worker` writes the workers.
Both use `INSERT OR IGNORE` — once a `run_id` exists, the row is
immutable through these helpers.

In [ ]:
run_id = "run-2026-05-20-demo"

ardb.upsert_run(
    filings,
    ardb.RunRecord(
        run_id=run_id,
        parent_run_id=None,
        country="US",
        started_at=now_iso(),
        finished_at=None,
        status="RUNNING",
        config=json.dumps({"source": "EDGAR", "max_workers": 3}),
        worker_count=3,
    ),
)

for i in (1, 2, 3):
    ardb.upsert_worker(
        filings,
        ardb.WorkerRecord(
            run_id=f"{run_id}-w{i}",
            parent_run_id=run_id,
            worker_id=i,
            started_at=now_iso(),
            finished_at=None,
            status="RUNNING",
        ),
    )

# Inspect what's in the table.
for row in filings.execute(
    "SELECT run_id, parent_run_id, worker_id, country, status, worker_count "
    "FROM scraper_runs ORDER BY worker_id IS NOT NULL, worker_id"
):
    print(dict(row))

**Try to re-upsert with a different status** — the original `RUNNING`
row must survive, because `upsert_run` is `INSERT OR IGNORE`:

In [ ]:
ardb.upsert_run(
    filings,
    ardb.RunRecord(
        run_id=run_id,
        parent_run_id=None,
        country="US",
        started_at=now_iso(),
        finished_at=now_iso(),
        status="SUCCESS",          # would-be update
        config=None,
        worker_count=99,
    ),
)

row = filings.execute(
    "SELECT status, finished_at, worker_count FROM scraper_runs WHERE run_id=?",
    (run_id,),
).fetchone()
print(dict(row))
print("→ original row preserved: status still 'RUNNING', finished_at still NULL")

## 3. Companies and filings

`upsert_company` and `upsert_filing` are both `INSERT OR IGNORE`. For
`filings`, the conflict key is the natural key
`(company_id, fiscal_year)` — a company can only have one filing per
fiscal year, and once recorded it is treated as immutable.

In [ ]:
ardb.upsert_company(
    filings,
    ardb.CompanyRecord(
        company_id="C001",
        name="Acme Corp",
        ticker="ACME",
        exchange="NASDAQ",
        country="US",
        updated_at=now_iso(),
    ),
)

ardb.upsert_company(
    filings,
    ardb.CompanyRecord(
        company_id="C002",
        name="Beta Industries",
        ticker="BETA",
        exchange="NYSE",
        country="US",
        updated_at=now_iso(),
    ),
)

# Generate stable UUIDs so the rest of the notebook can refer to them.
F_ACME_2023 = str(uuid.uuid4())
F_ACME_2024 = str(uuid.uuid4())
F_BETA_2024 = str(uuid.uuid4())

for filing_id, company, fy, fdate, rdate in [
    (F_ACME_2023, "C001", 2023, "2024-02-15", "2023-12-31"),
    (F_ACME_2024, "C001", 2024, "2025-02-15", "2024-12-31"),
    (F_BETA_2024, "C002", 2024, "2025-03-10", "2024-12-31"),
]:
    ardb.upsert_filing(
        filings,
        ardb.FilingRecord(
            filing_id=filing_id,
            company_id=company,
            fiscal_year=fy,
            filing_date=fdate,
            reporting_date=rdate,
            reporting_period="FY",
        ),
    )

print("filings rows:")
for r in filings.execute(
    "SELECT company_id, fiscal_year, filing_date FROM filings "
    "ORDER BY company_id, fiscal_year"
):
    print(" ", dict(r))

A second `upsert_filing` with the same `(company_id, fiscal_year)` but
a different `filing_id` is silently ignored:

In [ ]:
ardb.upsert_filing(
    filings,
    ardb.FilingRecord(
        filing_id=str(uuid.uuid4()),   # different UUID
        company_id="C001",
        fiscal_year=2024,              # same natural key
        filing_date="2099-01-01",
        reporting_date="2099-01-01",
        reporting_period="OTHER",
    ),
)

still_there = ardb.get_filing(filings, company_id="C001", fiscal_year=2024)
print(f"filing_id   : {still_there.filing_id}")
print(f"filing_date : {still_there.filing_date}")
print(f"period      : {still_there.reporting_period}")
print("→ original row preserved")

## 4. Filing files and the three statuses

A `filing_files` row has a `scrape_status` of `PENDING`, `SCRAPED`, or
`FAILED`. The natural key is `(filing_id, file_type, form_type)`.

We'll walk through a realistic lifecycle for the Acme 2023 filing:

1. The scraper records a `PENDING` row before attempting download.
2. The download fails — the row is updated to `FAILED`.
3. A retry succeeds — the row becomes `SCRAPED`.

Each step calls `upsert_filing_file`. Because the existing rows are
not yet `SCRAPED`, the helper just does an `INSERT OR REPLACE`.

In [ ]:
PDF_FILE_ID = str(uuid.uuid4())

# Step 1 — record the intent.
ardb.upsert_filing_file(
    filings,
    ardb.FilingFileRecord(
        file_id=PDF_FILE_ID,
        filing_id=F_ACME_2023,
        run_id=f"{run_id}-w1",
        worker_id=1,
        file_type="PDF",
        form_type="10-K",
        gcs_path=None,
        url="https://www.sec.gov/Archives/edgar/data/.../acme-2023.pdf",
        scrape_status="PENDING",
        scraped_at=None,
    ),
)
print("after step 1:",
      dict(ardb.get_filing_file(filings, F_ACME_2023, "PDF", "10-K").__dict__))

In [ ]:
# Step 2 — download failed, mark FAILED.
ardb.upsert_filing_file(
    filings,
    ardb.FilingFileRecord(
        file_id=PDF_FILE_ID,
        filing_id=F_ACME_2023,
        run_id=f"{run_id}-w1",
        worker_id=1,
        file_type="PDF",
        form_type="10-K",
        gcs_path=None,
        url="https://www.sec.gov/Archives/edgar/data/.../acme-2023.pdf",
        scrape_status="FAILED",
        scraped_at=None,
    ),
)
print("after step 2:",
      ardb.get_filing_file(filings, F_ACME_2023, "PDF", "10-K").scrape_status)

In [ ]:
# Step 3 — retry succeeded; FAILED → SCRAPED is allowed without force.
ardb.upsert_filing_file(
    filings,
    ardb.FilingFileRecord(
        file_id=PDF_FILE_ID,
        filing_id=F_ACME_2023,
        run_id=f"{run_id}-w1",
        worker_id=1,
        file_type="PDF",
        form_type="10-K",
        gcs_path="gs://my-bucket/C001/C001_10K_2023-12-31.pdf",
        url="https://www.sec.gov/Archives/edgar/data/.../acme-2023.pdf",
        scrape_status="SCRAPED",
        scraped_at=now_iso(),
    ),
)
row = ardb.get_filing_file(filings, F_ACME_2023, "PDF", "10-K")
print(f"status   : {row.scrape_status}")
print(f"gcs_path : {row.gcs_path}")

### `AlreadyScrapedError` — the SCRAPED row is now sticky

Once a row is `SCRAPED`, `upsert_filing_file` raises
`AlreadyScrapedError` rather than overwriting it. The scraper is
expected to call `get_filing_file()` first and skip the download — the
exception is a defence-in-depth signal for concurrent workers, not a
routine control-flow path.

In [ ]:
try:
    ardb.upsert_filing_file(
        filings,
        ardb.FilingFileRecord(
            file_id=str(uuid.uuid4()),
            filing_id=F_ACME_2023,
            run_id=f"{run_id}-w1",
            worker_id=1,
            file_type="PDF",
            form_type="10-K",
            gcs_path="gs://my-bucket/C001/replacement.pdf",
            url=None,
            scrape_status="SCRAPED",
            scraped_at=now_iso(),
        ),
    )
except ardb.AlreadyScrapedError as e:
    print(f"caught AlreadyScrapedError:\n  {e}")

### `force=True` — explicit re-scrape

When you really do want to overwrite a `SCRAPED` row (e.g. the GCS file
was deleted, or you want to refresh), pass `force=True`:

In [ ]:
NEW_FILE_ID = str(uuid.uuid4())

ardb.upsert_filing_file(
    filings,
    ardb.FilingFileRecord(
        file_id=NEW_FILE_ID,
        filing_id=F_ACME_2023,
        run_id=f"{run_id}-w1",
        worker_id=1,
        file_type="PDF",
        form_type="10-K",
        gcs_path="gs://my-bucket/C001/refreshed.pdf",
        url=None,
        scrape_status="SCRAPED",
        scraped_at=now_iso(),
    ),
    force=True,
)

row = ardb.get_filing_file(filings, F_ACME_2023, "PDF", "10-K")
print(f"file_id  : {row.file_id}   (replaced)")
print(f"gcs_path : {row.gcs_path}")

### Now scrape the XBRL too, plus the other filings

To exercise `get_scraped_pairs()` in the next section we need both PDF
**and** XBRL for some filings. We'll cover three deliberately different
states:

| Filing       | PDF       | XBRL      | Eligible for evaluation? |
|--------------|-----------|-----------|--------------------------|
| Acme 2023    | SCRAPED   | SCRAPED   | yes                      |
| Acme 2024    | SCRAPED   | PENDING   | no                       |
| Beta 2024    | FAILED    | SCRAPED   | no                       |

In [ ]:
def add_file(
    *, filing_id, file_type, status, form_type="10-K", gcs_filename=None
):
    ardb.upsert_filing_file(
        filings,
        ardb.FilingFileRecord(
            file_id=str(uuid.uuid4()),
            filing_id=filing_id,
            run_id=f"{run_id}-w1",
            worker_id=1,
            file_type=file_type,
            form_type=form_type,
            gcs_path=f"gs://my-bucket/{gcs_filename}" if gcs_filename else None,
            url=None,
            scrape_status=status,
            scraped_at=now_iso() if status == "SCRAPED" else None,
        ),
    )


# Acme 2023 — XBRL
add_file(
    filing_id=F_ACME_2023, file_type="XBRL",
    status="SCRAPED", gcs_filename="C001/C001_XBRL_2023-12-31.zip",
)

# Acme 2024 — PDF SCRAPED, XBRL PENDING
add_file(
    filing_id=F_ACME_2024, file_type="PDF",
    status="SCRAPED", gcs_filename="C001/C001_10K_2024-12-31.pdf",
)
add_file(
    filing_id=F_ACME_2024, file_type="XBRL", status="PENDING",
)

# Beta 2024 — PDF FAILED, XBRL SCRAPED
add_file(
    filing_id=F_BETA_2024, file_type="PDF", status="FAILED",
)
add_file(
    filing_id=F_BETA_2024, file_type="XBRL",
    status="SCRAPED", gcs_filename="C002/C002_XBRL_2024-12-31.zip",
)

# Snapshot of all filing_files rows.
print(f"{'filing_id':>10s}  {'file_type':<6s}  {'status':<8s}  gcs_path")
print("-" * 78)
for r in filings.execute(
    "SELECT filing_id, file_type, scrape_status, gcs_path "
    "FROM filing_files ORDER BY filing_id, file_type"
):
    fid_short = r["filing_id"][:8]
    print(f"{fid_short:>10s}  {r['file_type']:<6s}  "
          f"{r['scrape_status']:<8s}  {r['gcs_path'] or '-'}")

## 5. Reading back: simple lookups

`get_filing` accepts either a `filing_id` or the natural key
`(company_id, fiscal_year)`. `list_companies` returns every row
ordered by `company_id`.

In [ ]:
print("By filing_id:")
print(" ", ardb.get_filing(filings, filing_id=F_ACME_2023))

print("\nBy natural key:")
print(" ", ardb.get_filing(filings, company_id="C001", fiscal_year=2024))

print("\nMissing:")
print(" ", ardb.get_filing(filings, filing_id="does-not-exist"))

print("\nCompanies:")
for c in ardb.list_companies(filings):
    print(f"  {c.company_id}  {c.ticker:6s}  {c.name}")

## 6. `get_scraped_pairs()` — the evaluator's entry point

This is the most important read in the module. It returns every filing
for which **both** the PDF and the XBRL are `SCRAPED` — i.e. ready to
evaluate. Anything missing one side is excluded by the inner joins;
there is no Python-side filtering.

Given the table we set up above, only **Acme 2023** should come back.

In [ ]:
pairs = ardb.get_scraped_pairs(filings)

print(f"{len(pairs)} pair(s) ready for evaluation:")
for p in pairs:
    print(f"  filing_id  : {p.filing_id}")
    print(f"  company    : {p.company_id} / FY{p.fiscal_year}")
    print(f"  PDF        : {p.pdf_gcs_path}")
    print(f"  XBRL       : {p.xbrl_gcs_path}")

The optional `company_id` and `fiscal_year` filters compose with AND.
With no PDF/XBRL pair for Acme 2024 or Beta 2024, narrowing returns
nothing:

In [ ]:
print("company_id='C001'         :", ardb.get_scraped_pairs(filings, company_id="C001"))
print("fiscal_year=2024          :", ardb.get_scraped_pairs(filings, fiscal_year=2024))
print("C001 + 2023               :", len(ardb.get_scraped_pairs(filings, company_id="C001", fiscal_year=2023)))

**What if we complete the missing XBRL for Acme 2024?** It should
immediately appear in `get_scraped_pairs()`:

In [ ]:
add_file(
    filing_id=F_ACME_2024, file_type="XBRL",
    status="SCRAPED", gcs_filename="C001/C001_XBRL_2024-12-31.zip",
)

for p in ardb.get_scraped_pairs(filings):
    print(f"  {p.company_id} / FY{p.fiscal_year}  ({p.filing_id[:8]}…)")

## 7. Writing evaluation metrics

The evaluator side has two helpers:

- `write_evaluation` — one row per evaluator run on one filing.
- `write_evaluation_scores_by_statement` — per-statement breakdown,
  FK'd to the parent evaluation row.

Both helpers use plain `INSERT` (not `INSERT OR REPLACE`), so a
duplicate `evaluation_id` raises `IntegrityError`. This makes
`metrics.db` append-only by default — use a fresh UUID per evaluator
run.

`scope`, `output_json`, and `output_diff` are stored as **already
serialised strings**; the caller does the `json.dumps`.

In [ ]:
eval_id = str(uuid.uuid4())

ardb.write_evaluation(
    metrics,
    ardb.EvaluationRecord(
        evaluation_id=eval_id,
        filing_id=F_ACME_2023,
        company_id="C001",
        fiscal_year=2023,
        evaluated_at=now_iso(),
        pdf_source="gs://my-bucket/C001/C001_10K_2023-12-31.pdf",
        xbrl_source="gs://my-bucket/C001/C001_XBRL_2023-12-31.zip",
        scope=json.dumps(["IncomeStatement", "BalanceSheet", "CashFlow"]),
        xbrl_facts_scope=120,
        pdf_facts_scope=110,
        matched=100,
        missed=10,
        spurious=5,
        tier1_matches=80,
        tier2_matches=20,
        coverage=0.91,
        precision=0.95,
        recall=0.90,
        f1=0.92,
        exact_match_rate=0.85,
        within_1pct_rate=0.93,
        within_5pct_rate=0.97,
        output_json=json.dumps({"summary": "ok"}),
        output_diff="(diff text would go here)",
    ),
)

ardb.write_evaluation_scores_by_statement(
    metrics,
    [
        ardb.StatementScoreRecord(
            evaluation_id=eval_id,
            statement="IncomeStatement",
            coverage=0.90, precision=0.92, recall=0.91, f1=0.915,
            exact_match_rate=0.85, within_1pct_rate=0.93, within_5pct_rate=0.97,
        ),
        ardb.StatementScoreRecord(
            evaluation_id=eval_id,
            statement="BalanceSheet",
            coverage=0.88, precision=0.90, recall=0.89, f1=0.895,
            exact_match_rate=0.83, within_1pct_rate=0.92, within_5pct_rate=0.96,
        ),
        ardb.StatementScoreRecord(
            evaluation_id=eval_id,
            statement="CashFlow",
            coverage=0.95, precision=0.97, recall=0.94, f1=0.955,
            exact_match_rate=0.90, within_1pct_rate=0.96, within_5pct_rate=0.99,
        ),
    ],
)

print(f"wrote evaluation {eval_id}")

In [ ]:
# Read back the headline numbers.
row = metrics.execute(
    "SELECT company_id, fiscal_year, matched, missed, f1, coverage "
    "FROM evaluations WHERE evaluation_id = ?",
    (eval_id,),
).fetchone()
print("evaluation row:", dict(row))

print("\nper-statement scores:")
for r in metrics.execute(
    "SELECT statement, coverage, f1 FROM evaluation_scores_by_statement "
    "WHERE evaluation_id = ? ORDER BY statement",
    (eval_id,),
):
    print(f"  {r['statement']:<16s}  coverage={r['coverage']:.3f}  f1={r['f1']:.3f}")

**A duplicate `evaluation_id` is rejected** — by design:

In [ ]:
try:
    ardb.write_evaluation(
        metrics,
        ardb.EvaluationRecord(
            evaluation_id=eval_id,   # same UUID
            filing_id=F_ACME_2023, company_id="C001", fiscal_year=2023,
            evaluated_at=now_iso(),
            pdf_source=None, xbrl_source=None, scope=None,
            xbrl_facts_scope=None, pdf_facts_scope=None,
            matched=None, missed=None, spurious=None,
            tier1_matches=None, tier2_matches=None,
            coverage=None, precision=None, recall=None, f1=None,
            exact_match_rate=None, within_1pct_rate=None, within_5pct_rate=None,
            output_json=None, output_diff=None,
        ),
    )
except sqlite3.IntegrityError as e:
    print(f"caught IntegrityError:\n  {e}")

## 8. Foreign-key enforcement, briefly

FK pragma is on, so writing a `filing_files` row that points at a
non-existent `filing_id` is rejected at insert time:

In [ ]:
try:
    ardb.upsert_filing_file(
        filings,
        ardb.FilingFileRecord(
            file_id=str(uuid.uuid4()),
            filing_id="does-not-exist",       # bogus FK target
            run_id=f"{run_id}-w1",
            worker_id=1,
            file_type="PDF",
            form_type="10-K",
            gcs_path=None, url=None,
            scrape_status="PENDING", scraped_at=None,
        ),
    )
except sqlite3.IntegrityError as e:
    print(f"caught IntegrityError:\n  {e}")

## 9. Teardown

Close both connections. Once this notebook's kernel shuts down, the
`tempfile.TemporaryDirectory` is destroyed and both databases vanish
with it — nothing else on disk was touched.

In [ ]:
filings.close()
metrics.close()

print(f"Files still present in temp dir (will be deleted on kernel shutdown):")
for p in sorted(WORKDIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size} bytes)")

---

### Cheatsheet — upsert behaviour

| Helper                 | Conflict key                            | Behaviour                                                    |
|------------------------|-----------------------------------------|--------------------------------------------------------------|
| `upsert_run`           | `run_id`                                | `INSERT OR IGNORE` — first row wins                          |
| `upsert_worker`        | `run_id`                                | `INSERT OR IGNORE` — first row wins                          |
| `upsert_company`       | `company_id`                            | `INSERT OR IGNORE`                                           |
| `upsert_filing`        | `(company_id, fiscal_year)`             | `INSERT OR IGNORE` — filing event is immutable               |
| `upsert_filing_file`   | `(filing_id, file_type, form_type)`     | replace if not yet SCRAPED; else raise `AlreadyScrapedError` (or replace with `force=True`) |
| `write_evaluation`     | `evaluation_id`                         | `INSERT` only — duplicate raises `IntegrityError`            |